<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Computer%20Vision/Vision%20Transformers%20(ViT)%20Alapok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vision Transformers (ViT) Alapok

A **Vision Transformer (ViT)** a Transformer architektúrát alkalmazza képekre. Az NLP-ben bevált self-attention mechanizmust használja konvolúció helyett.

## Fő ötlet

A képet **patch-ekre** (kis négyzetes darabokra) vágjuk, és ezeket tokenekként kezeljük - pont mint a szavakat az NLP-ben!

```
Kép (224×224) → 16×16 patch-ek → 196 token → Transformer → Osztályozás
```

## Tartalomjegyzék

1. ViT architektúra
2. Patch embedding
3. Self-attention képekre
4. Implementáció
5. ViT vs CNN összehasonlítás

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
torch.manual_seed(42)

## 1. ViT architektúra

### Az eredeti Transformer → ViT adaptáció

| NLP Transformer | Vision Transformer |
|-----------------|--------------------|
| Szó tokenek | Patch tokenek |
| Word embedding | Patch embedding |
| Positional encoding | Learnable position embedding |
| [CLS] token | [CLS] token (osztályozáshoz) |

### ViT folyamat

```
┌─────────────────────────────────────────────────────────────┐
│  Kép (H×W×C)                                                │
│       ↓                                                     │
│  Patch-ekre vágás (P×P méretű)  →  N = (H/P) × (W/P) patch  │
│       ↓                                                     │
│  Lineáris projekció  →  [N × D] embedding                   │
│       ↓                                                     │
│  + [CLS] token + Positional embedding                       │
│       ↓                                                     │
│  Transformer Encoder (L réteg)                              │
│       ↓                                                     │
│  [CLS] token kimenete → MLP Head → Osztály                  │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# Patch-elés vizualizáció
def visualize_patches(img_size=224, patch_size=16):
    # Teszt kép
    img = Image.new('RGB', (img_size, img_size), (240, 245, 250))
    draw = ImageDraw.Draw(img)
    draw.ellipse([40, 40, 180, 180], fill=(200, 80, 80))
    draw.rectangle([100, 100, 200, 200], fill=(80, 150, 80))

    n_patches = img_size // patch_size

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))

    # Eredeti kép
    axes[0].imshow(img)
    axes[0].set_title(f'Eredeti kép\n{img_size}×{img_size}', fontsize=11)
    axes[0].axis('off')

    # Rács overlay
    axes[1].imshow(img)
    for i in range(n_patches + 1):
        axes[1].axhline(i * patch_size, color='blue', linewidth=1)
        axes[1].axvline(i * patch_size, color='blue', linewidth=1)
    axes[1].set_title(f'Patch rács\n{n_patches}×{n_patches} = {n_patches**2} patch', fontsize=11)
    axes[1].axis('off')

    # Néhány patch kiemelve
    img_array = np.array(img)
    patches_to_show = [(0, 0), (3, 3), (7, 10), (12, 8)]

    axes[2].set_title('Kiválasztott patch-ek', fontsize=11)
    for idx, (pi, pj) in enumerate(patches_to_show):
        patch = img_array[pi*patch_size:(pi+1)*patch_size,
                         pj*patch_size:(pj+1)*patch_size]
        ax_inset = axes[2].inset_axes([0.25*idx, 0.3, 0.22, 0.4])
        ax_inset.imshow(patch)
        ax_inset.set_title(f'({pi},{pj})', fontsize=9)
        ax_inset.axis('off')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    print(f"Patch méret: {patch_size}×{patch_size}")
    print(f"Patch-ek száma: {n_patches}×{n_patches} = {n_patches**2}")
    print(f"Egy patch vektor mérete: {patch_size}×{patch_size}×3 = {patch_size*patch_size*3}")

visualize_patches()

## 2. Patch Embedding

A patch-eket **lineáris projekcióval** alakítjuk embedding vektorokká.

$$\mathbf{z}_0 = [\mathbf{x}_{class}; \mathbf{x}_1^p\mathbf{E}; \mathbf{x}_2^p\mathbf{E}; ...; \mathbf{x}_N^p\mathbf{E}] + \mathbf{E}_{pos}$$

ahol:
- $\mathbf{x}_i^p$: i-edik patch flatten-elve
- $\mathbf{E}$: Patch embedding mátrix
- $\mathbf{E}_{pos}$: Pozíció embedding

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Kép → Patch tokenek konverzió.

    Megvalósítás: Conv2d stride=patch_size (hatékonyabb mint reshape+linear)
    """
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2

        # Conv2d a patch-eléshez és projekcióhoz egyben
        self.projection = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x):
        # x: [B, C, H, W]
        x = self.projection(x)  # [B, embed_dim, H/P, W/P]
        x = x.flatten(2)        # [B, embed_dim, n_patches]
        x = x.transpose(1, 2)   # [B, n_patches, embed_dim]
        return x

# Teszt
patch_embed = PatchEmbedding(img_size=224, patch_size=16, embed_dim=768)
dummy_img = torch.randn(2, 3, 224, 224)
patches = patch_embed(dummy_img)

print(f"Bemenet: {dummy_img.shape}")
print(f"Patch embedding: {patches.shape}")
print(f"  → {patches.shape[1]} patch, {patches.shape[2]} dimenziós embedding")

## 3. Self-Attention képekre

A self-attention minden patch-et minden más patch-hez viszonyít:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Előny**: Globális receptív mező már az első rétegben!

**Hátrány**: $O(N^2)$ komplexitás a patch-ek számában

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention."""
    def __init__(self, embed_dim=768, n_heads=12, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape

        # Q, K, V projekció
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # [3, B, heads, N, head_dim]
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)

        # Output
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x, attn

# Teszt
mha = MultiHeadAttention(embed_dim=768, n_heads=12)
x = torch.randn(2, 197, 768)  # 196 patch + 1 CLS token
out, attn = mha(x)

print(f"Input: {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention weights: {attn.shape}")
print(f"  → {attn.shape[1]} head, {attn.shape[2]}×{attn.shape[3]} attention mátrix")

In [ ]:
# Attention vizualizáció
def visualize_attention(attn_weights, img_size=224, patch_size=16):
    """CLS token attention vizualizáció."""
    n_patches_side = img_size // patch_size

    # CLS token attention az összes patch-re (első sor, 1: -tól mert 0 a CLS)
    cls_attn = attn_weights[0, :, 0, 1:].detach().numpy()  # [heads, n_patches]

    fig, axes = plt.subplots(2, 6, figsize=(15, 5))

    for i, ax in enumerate(axes.flat):
        if i < 12:
            attn_map = cls_attn[i].reshape(n_patches_side, n_patches_side)
            im = ax.imshow(attn_map, cmap='viridis')
            ax.set_title(f'Head {i+1}', fontsize=9)
            ax.axis('off')

    plt.suptitle('CLS token attention különböző head-ekben', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_attention(attn)

## 4. Teljes ViT implementáció

In [ ]:
class TransformerBlock(nn.Module):
    """Egy Transformer encoder blokk."""
    def __init__(self, embed_dim=768, n_heads=12, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * mlp_ratio, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # Pre-norm + residual
        attn_out, _ = self.attn(self.norm1(x))
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

class VisionTransformer(nn.Module):
    """Vision Transformer (ViT)."""
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 n_classes=1000, embed_dim=768, depth=12, n_heads=12):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches

        # CLS token és position embedding
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))

        # Transformer blokkok
        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, n_heads) for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, n_classes)

        # Inicializálás
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]

        # Patch embedding
        x = self.patch_embed(x)  # [B, n_patches, embed_dim]

        # CLS token hozzáadása
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # [B, n_patches+1, embed_dim]

        # Position embedding
        x = x + self.pos_embed

        # Transformer blokkok
        x = self.blocks(x)
        x = self.norm(x)

        # Osztályozás a CLS tokenből
        return self.head(x[:, 0])

# Modell létrehozása
vit = VisionTransformer(
    img_size=224, patch_size=16,
    n_classes=10, embed_dim=384, depth=6, n_heads=6
)

# Teszt
x = torch.randn(4, 3, 224, 224)
out = vit(x)

print(f"Input: {x.shape}")
print(f"Output: {out.shape}")
print(f"\nParaméterek: {sum(p.numel() for p in vit.parameters()):,}")

## 5. ViT vs CNN összehasonlítás

| Szempont | CNN (ResNet) | ViT |
|----------|--------------|-----|
| Receptív mező | Fokozatosan növekszik | Globális az 1. rétegtől |
| Inductive bias | Lokalitás, transzláció invariancia | Kevés (pozíció embedding) |
| Adat igény | Közepes | Nagy (vagy pretrain!) |
| Skálázhatóság | Korlátozott | Kiváló |
| Kis adaton | Jobb | Rosszabb (túltanulás) |

In [ ]:
# Pozíció embedding vizualizáció
def visualize_position_embedding(pos_embed, patch_size=16, img_size=224):
    """Pozíció embedding hasonlóság vizualizálása."""
    n_patches_side = img_size // patch_size

    # CLS token nélkül
    pos = pos_embed[0, 1:, :].detach().numpy()  # [n_patches, embed_dim]

    # Cosine hasonlóság
    pos_norm = pos / np.linalg.norm(pos, axis=1, keepdims=True)
    sim = pos_norm @ pos_norm.T

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Teljes hasonlóság mátrix
    im = axes[0].imshow(sim, cmap='RdBu_r', vmin=-1, vmax=1)
    axes[0].set_title('Pozíció embedding hasonlóság\n(összes patch pár)', fontsize=11)
    axes[0].set_xlabel('Patch index')
    axes[0].set_ylabel('Patch index')
    plt.colorbar(im, ax=axes[0])

    # Kiválasztott patch-ek hasonlósága
    ref_patches = [0, 97, 195]  # bal felső, közép, jobb alsó

    for i, ref in enumerate(ref_patches):
        ref_sim = sim[ref].reshape(n_patches_side, n_patches_side)
        ax_inset = axes[1].inset_axes([0.33*i, 0.1, 0.3, 0.8])
        ax_inset.imshow(ref_sim, cmap='RdBu_r', vmin=-1, vmax=1)
        row, col = ref // n_patches_side, ref % n_patches_side
        ax_inset.set_title(f'Patch ({row},{col})', fontsize=9)
        ax_inset.axis('off')

    axes[1].axis('off')
    axes[1].set_title('Hasonlóság referencia patch-ekhez', fontsize=11)

    plt.tight_layout()
    plt.show()

visualize_position_embedding(vit.pos_embed)

## Összefoglalás

### ViT lényege

1. **Patch-elés**: Kép → 16×16 patch-ek
2. **Lineáris projekció**: Patch → Embedding vektor
3. **CLS token**: Speciális token az osztályozáshoz
4. **Transformer**: Self-attention + MLP blokkok

### ViT variánsok

| Modell | Patch | Embed dim | Depth | Heads | Params |
|--------|-------|-----------|-------|-------|--------|
| ViT-Ti | 16 | 192 | 12 | 3 | 5.7M |
| ViT-S | 16 | 384 | 12 | 6 | 22M |
| ViT-B | 16 | 768 | 12 | 12 | 86M |
| ViT-L | 16 | 1024 | 24 | 16 | 307M |

### Használat PyTorch-ban

```python
from torchvision.models import vit_b_16, ViT_B_16_Weights

model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
```